# Transformer Encoder



```
                  Input
                    │
                    ▼
             Word Embedding
                    │
                    ▼
          Positional Encoding
                    │
                    ▼
      ┌─────────────────────────┐
      │     Encoder Block 1     │
      └─────────────────────────┘
                    │
                    ▼
      ┌─────────────────────────┐
      │     Encoder Block 2     │
      └─────────────────────────┘
                    │
                    ▼
      ┌─────────────────────────┐
      │     Encoder Block 3     │
      └─────────────────────────┘
                    │
                   ...
                    │
                    ▼
      ┌─────────────────────────┐
      │     Encoder Block N     │
      └─────────────────────────┘
                    │
                    ▼
               Encoder Output
```



##Positional Encoding

In [29]:
import math
import torch
import torch.nn as nn

In [30]:
class PositionalEncoding(nn.Module):
  def __init__(self,d_model,max_len=5000):
    super().__init__()

    position=torch.arange(max_len).unsqueeze(1)

    div_term=torch.exp(
        torch.arange(0,d_model,2)*
        (-math.log(10000.0)/d_model)
    )

    pe=torch.zeros(max_len,d_model)

    pe[:,0::2]=torch.sin(position*div_term)
    pe[:,1::2]=torch.cos(position*div_term)

    pe=pe.unsqueeze(0) #add dimension for batch

    # Register as a buffer so it moves to GPU automatically but is not trained
    self.register_buffer('pe',pe)

  def forward(self,x):
    x=x+self.pe[:,:x.size(1)]
    return x


## Multihead Attention

In [31]:
class MultiHeadAttention(nn.Module):
  def __init__(self,embed_dim,num_heads):
    super().__init__()

    self.embed_dim=embed_dim
    self.num_heads=num_heads
    self.head_dim=embed_dim//num_heads

    self.query=nn.Linear(embed_dim,embed_dim,bias=False)
    self.key=nn.Linear(embed_dim,embed_dim,bias=False)
    self.value=nn.Linear(embed_dim,embed_dim,bias=False)

    self.fc_out=nn.Linear(embed_dim,embed_dim)

  def forward(self,x):
    print("MultiHead Attention:\n")
    B,S,E=x.shape   #(batch,seq_len,embed_dim)

    Q=self.query(x) #(batch,seq_len,embed_dim)
    K=self.key(x)   #(batch,seq_len,embed_dim)
    V=self.value(x) #(batch,seq_len,embed_dim)

    print("Shape of Q:",Q.shape)
    print("="*60)

    # split into heads
    Q=Q.view(B,S,self.num_heads,self.head_dim)
    K=K.view(B,S,self.num_heads,self.head_dim)
    V=V.view(B,S,self.num_heads,self.head_dim)
    # shape of Q,K,V (batch,seq_len,numOfheads,head_dim)
    print("\nAfter View The shape of Q:",Q.shape)
    print("="*60)

    Q=Q.transpose(1,2)
    K=K.transpose(1,2)
    V=V.transpose(1,2)
    #after transpose shape of Q,K,V (batch,numOfHeads,seq_len,head_dim)
    print("\nAfter Transpose The shape of Q:",Q.shape)
    print("="*60)

    # scores
    scores=torch.matmul(
        Q,K.transpose(-2,-1)
    )
    # print("\nScores: ",scores.shape)

    # scaling
    scores=scores/math.sqrt(self.head_dim)

    # attention weight
    attention=torch.softmax(
        scores,
        dim=-1
    )

    # context
    context=torch.matmul(
        attention,
        V
    )

    print("\nContext Shape :",context.shape)
    print("="*50)

    context=context.transpose(1,2)
    # before transpose shape of context(B,H,S,D)
    # after transpose shape of context(B,S,H,D)

    context=context.contiguous()
    # Merge
    context=context.view(
        B,S,self.embed_dim
    )
    print("\nMerged context shape :",context.shape)
    print("="*50)

    output=self.fc_out(context)
    print("\nOutput Shape :",output.shape)
    print("="*50)
    return output

## LayerNormalization

In [32]:
class MyLayerNorm(nn.Module):
  def __init__(self,d_model,eps=1e-5):
    super().__init__()

    self.gamma=nn.Parameter(torch.ones(d_model))
    self.beta=nn.Parameter(torch.zeros(d_model))
    self.eps=eps

  def forward(self,x):
    print("="*50)
    print("\nLayer Normalization:\n")
    # print("Input")
    # print(x)
    print("Input Shape:",x.shape)

    mean=x.mean(dim=-1,keepdim=True)
    print("="*50)
    # print("\nMean")
    # print(mean)
    print("Mean Shape:",mean.shape)

    var=x.var(dim=-1,keepdim=True)
    print("="*50)
    # print("\nVariance")
    # print(var)
    print("Variance Shape:",var.shape)

    x_hat=(x-mean)/torch.sqrt(var+self.eps)
    print("="*50)
    # print("\nNormalized Input")
    # print(x_hat)
    print("\nX_hat Shape",x_hat.shape)

    out=self.gamma*x_hat+self.beta
    print("="*50)
    # print("\nOutput")
    # print(out)
    print("layerNorm Output Shape:",out.shape)

    return out

## FeedForward Network

In [33]:
class FeedForward(nn.Module):
  def __init__(self,d_model,hidden_dim):
    super().__init__()

    self.fc1=nn.Linear(d_model,hidden_dim)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(
        hidden_dim,d_model
    )

  def forward(self,x):
    out=self.fc1(x)
    out=self.relu(out)
    out=self.fc2(out)
    return out

## Encoder Block

In [34]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,hidden_dim):
    super().__init__()

    self.attention=MultiHeadAttention(embed_dim,num_heads)
    self.norm1=MyLayerNorm(embed_dim)
    self.feed_forward=FeedForward(embed_dim,hidden_dim)
    self.norm2=MyLayerNorm(embed_dim)

  def forward(self,x):
    attention_output=self.attention(x)
    x=x+attention_output
    x=self.norm1(x)

    feed_forward_output=self.feed_forward(x)
    x=x+feed_forward_output
    x=self.norm2(x)
    return x

# Transformer Encoder (Multi Layer)

In [35]:
class TransformerEncoder(nn.Module):
  def __init__(self,vocab_size,d_model,num_heads,hidden_dim,num_layers,max_len):
    super().__init__()

    self.embedding=nn.Embedding(
        vocab_size,
        d_model
    )
    self.position=PositionalEncoding(
        d_model,
        max_len
    )

    self.layers=nn.ModuleList(
        [
            TransformerEncoderBlock(
                d_model,
                num_heads,
                hidden_dim
            )
            for _ in range(num_layers)
        ]
    )

  def forward(self,x):
    print("="*50)
    print("Input IDs:", x.shape)
    x=self.embedding(x)
    print("After Embedding:", x.shape)
    x=self.position(x)
    print("After Position Encoding:", x.shape)

    for i,layer in enumerate(self.layers):
      x=layer(x)
      print(f"\nAfter Encoder Layer {i+1}: {x.shape}")


    return x



```
Embedding

+

Positional Encoding

+

N × Encoder Block
```



In [36]:
VOCAB_SIZE = 100
D_MODEL = 8
NUM_HEADS = 2
HIDDEN_DIM = 32
NUM_LAYERS = 3
MAX_LEN = 20


## Create Model

In [37]:
encoder = TransformerEncoder(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN
)

print(encoder)

TransformerEncoder(
  (embedding): Embedding(100, 8)
  (position): PositionalEncoding()
  (layers): ModuleList(
    (0-2): 3 x TransformerEncoderBlock(
      (attention): MultiHeadAttention(
        (query): Linear(in_features=8, out_features=8, bias=False)
        (key): Linear(in_features=8, out_features=8, bias=False)
        (value): Linear(in_features=8, out_features=8, bias=False)
        (fc_out): Linear(in_features=8, out_features=8, bias=True)
      )
      (norm1): MyLayerNorm()
      (feed_forward): FeedForward(
        (fc1): Linear(in_features=8, out_features=32, bias=True)
        (relu): ReLU()
        (fc2): Linear(in_features=32, out_features=8, bias=True)
      )
      (norm2): MyLayerNorm()
    )
  )
)


In [38]:
x = torch.tensor([
    [5, 8, 10, 2],
    [7, 1, 9, 4]
])
print(x.shape)

torch.Size([2, 4])


In [39]:
output = encoder(x)
print()
print('='*50)
print('='*50)
print('='*50)
print(output)
print("Final Output Shape")
print(output.shape)

Input IDs: torch.Size([2, 4])
After Embedding: torch.Size([2, 4, 8])
After Position Encoding: torch.Size([2, 4, 8])
MultiHead Attention:

Shape of Q: torch.Size([2, 4, 8])

After View The shape of Q: torch.Size([2, 4, 2, 4])

After Transpose The shape of Q: torch.Size([2, 2, 4, 4])

Context Shape : torch.Size([2, 2, 4, 4])

Merged context shape : torch.Size([2, 4, 8])

Output Shape : torch.Size([2, 4, 8])

Layer Normalization:

Input Shape: torch.Size([2, 4, 8])
Mean Shape: torch.Size([2, 4, 1])
Variance Shape: torch.Size([2, 4, 1])

X_hat Shape torch.Size([2, 4, 8])
layerNorm Output Shape: torch.Size([2, 4, 8])

Layer Normalization:

Input Shape: torch.Size([2, 4, 8])
Mean Shape: torch.Size([2, 4, 1])
Variance Shape: torch.Size([2, 4, 1])

X_hat Shape torch.Size([2, 4, 8])
layerNorm Output Shape: torch.Size([2, 4, 8])

After Encoder Layer 1: torch.Size([2, 4, 8])
MultiHead Attention:

Shape of Q: torch.Size([2, 4, 8])

After View The shape of Q: torch.Size([2, 4, 2, 4])

After Transpo

#Dimension Flow



```
Input IDs
(2,4)
      │
      ▼
Embedding
(2,4,8)
      │
      ▼
Positional Encoding
(2,4,8)
      │
      ▼
Encoder Block-1
      │
      ├── MultiHeadAttention
      │      (2,4,8)
      │
      ├── Residual
      │      (2,4,8)
      │
      ├── LayerNorm
      │      (2,4,8)
      │
      ├── FeedForward
      │      (2,4,8)
      │
      ├── Residual
      │      (2,4,8)
      │
      └── LayerNorm
             (2,4,8)
      │
      ▼
Encoder Block-2
      │
      ▼
Encoder Block-3
      │
      ▼
Final Output
(2,4,8)
```

